# 05. Multi-batch와 Python/Torch interface

cuPF의 Python API는 두 층으로 볼 수 있습니다.

- `_cupf` pybind: `NewtonOptions`, `NewtonSolver.initialize`, `solve`, `solve_batch`
- `cupf.solve(...)`: Torch tensor를 받아 autograd graph에 연결하는 helper

여기서는 작은 `case9`로 API 모양을 확인합니다. 대형 case 성능 측정은 앞 노트북의 benchmark runner가 담당합니다.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'cuPF').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from python.tutorial import tutorial_utils as tu

plt.rcParams['figure.figsize'] = (8, 4.8)
plt.rcParams['axes.grid'] = False

RUN_LIVE = True


In [2]:
try:
    import torch
    has_cuda = torch.cuda.is_available()
except Exception:
    has_cuda = False

case = tu.load_case('case9')
cupf_cpu = None
cupf_gpu = None
if has_cuda:
    print('CUDA is available; building GPU artifacts first so solve_batch can use real multi-batch.')
    gpu_build = tu.build_eval('gpu', jobs=2, timeout=3600)
    print(tu.command_summary(gpu_build))
    cupf_gpu, gpu_message = tu.import_cupf_from_build('gpu')
    print(gpu_message)
else:
    print('CUDA is not available; building CPU artifacts for API-shape demonstration.')
    cpu_build = tu.build_eval('cpu', jobs=2, timeout=2400)
    print(tu.command_summary(cpu_build))
    cupf_cpu, cpu_message = tu.import_cupf_from_build('cpu')
    print(cpu_message)


CUDA is available; building GPU artifacts first so solve_batch can use real multi-batch.


$ bash benchmark/scripts/build_eval.bash gpu --jobs 2
[OK] elapsed=0.3s
[build_eval] configure (gpu) -> /workspace/gpu-powerflow-master/cuPF/build/eval-gpu
-- spdlog not found: disabling cuPF logging
-- cuDSS reordering algorithm: DEFAULT
-- cuDSS MT mode: OFF
-- cuDSS host threads: AUTO
-- cuDSS ND_NLEVELS: AUTO
-- Found pybind11: /usr/local/include (found version "3.0.1")
-- Configuring done
-- Generating done
-- Build files have been written to: /workspace/gpu-powerflow-master/cuPF/build/eval-gpu
[build_eval] build _cupf + cupf_cpp_evaluate (-j 2)
Consolidate compiler generated dependencies of target cupf
[ 89%] Built target cupf
Consolidate compiler generated dependencies of target _cupf
[100%] Built target _cupf
[ 86%] Built target cupf
Consolidate compiler generated dependencies of target cupf_cpp_evaluate
[100%] Built target cupf_cpp_evaluate
[build_eval] done. artifacts under /workspace/gpu-powerflow-master/cuPF/build/eval-gpu
[build_eval]   /workspace/gpu-powerflow-master/cuPF

In [3]:
try:
    import torch
    has_cuda = torch.cuda.is_available()
except Exception:
    has_cuda = False

cupf = cupf_gpu if has_cuda and cupf_gpu is not None else cupf_cpu
kind = 'gpu' if has_cuda and cupf_gpu is not None else 'cpu'
if cupf is None:
    print('solve_batch skipped because cupf import failed.')
else:
    try:
        opts = cupf.NewtonOptions()
        if kind == 'gpu':
            opts.backend = cupf.BackendKind.CUDA
            opts.compute = cupf.ComputePolicy.Mixed
            scales = np.array([1.00, 1.01, 0.99, 1.02])
        else:
            opts.backend = cupf.BackendKind.CPU
            opts.compute = cupf.ComputePolicy.FP64
            opts.cpu_linear_solver = cupf.CpuLinearSolverKind.KLU
            opts.cpu_jacobian = cupf.CpuJacobianKind.Native
            scales = np.array([1.00])
            print('CPU solve_batch currently supports batch_size=1; real multi-batch is demonstrated on CUDA Mixed when available.')
        solver = cupf.NewtonSolver(opts)
        solver.initialize(case.ybus.indptr, case.ybus.indices, case.ybus.data, case.ybus.shape[0], case.ybus.shape[1], case.pv, case.pq)
        sbus_batch = np.stack([case.sbus * scale for scale in scales]).astype(np.complex128)
        v0_batch = np.stack([case.v0 for _ in scales]).astype(np.complex128)
        config = cupf.NRConfig()
        config.tolerance = 1e-6 if kind == 'gpu' else 1e-8
        config.max_iter = 50
        result = solver.solve_batch(
            case.ybus.indptr, case.ybus.indices, case.ybus.data,
            case.ybus.shape[0], case.ybus.shape[1],
            sbus_batch, v0_batch, case.pv, case.pq, config,
        )
        display(pd.DataFrame({
            'scenario': np.arange(result.batch_size),
            'load_scale': scales,
            'backend': kind,
            'converged': result.converged_numpy.astype(bool),
            'iterations': result.iterations_numpy,
            'final_mismatch': result.final_mismatch_numpy,
        }))
        print('V_numpy shape:', result.V_numpy.shape)
    except Exception as exc:
        print(f'solve_batch skipped/error: {type(exc).__name__}: {exc}')
        print('Note: CPU build supports the API shape, while batch_size > 1 currently requires CUDA FP32/Mixed.')


,scenario,load_scale,backend,converged,iterations,final_mismatch
0,0,1.00,gpu,True,4,3.424897e-07
1,1,1.01,gpu,True,4,3.515191e-07
2,2,0.99,gpu,True,4,3.322251e-07
3,3,1.02,gpu,True,4,3.641224e-07


V_numpy shape: (4, 9)


## Torch autograd

Torch path는 CUDA build와 Torch extension method가 필요합니다. 사용할 수 있으면 load perturbation `load_p`, `load_q`에 대한 gradient가 backward로 계산됩니다.


In [4]:
try:
    import torch
    if not torch.cuda.is_available():
        print('Torch autograd skipped: CUDA is not available.')
    else:
        cupf_gpu = cupf if kind == 'gpu' else tu.import_cupf_from_build('gpu')[0]
        if cupf_gpu is None or getattr(cupf_gpu, 'solve', None) is None or not hasattr(cupf_gpu.NewtonSolver, 'solve_with_adjoint_cache_torch'):
            print('Torch autograd skipped: cuPF was not built with Torch extension methods.')
        else:
            opts = cupf_gpu.NewtonOptions()
            opts.backend = cupf_gpu.BackendKind.CUDA
            opts.compute = cupf_gpu.ComputePolicy.Mixed
            solver = cupf_gpu.NewtonSolver(opts)
            solver.initialize(case.ybus.indptr, case.ybus.indices, case.ybus.data, case.ybus.shape[0], case.ybus.shape[1], case.pv, case.pq)
            device = torch.device('cuda')
            dtype = torch.float32
            load_p = torch.zeros((2, case.ybus.shape[0]), device=device, dtype=dtype, requires_grad=True)
            load_q = torch.zeros((2, case.ybus.shape[0]), device=device, dtype=dtype, requires_grad=True)
            load_p.data[1] = 0.01
            sbus_base_re = torch.tensor(case.sbus.real, device=device, dtype=dtype)
            sbus_base_im = torch.tensor(case.sbus.imag, device=device, dtype=dtype)
            v0_va = torch.angle(torch.tensor(case.v0, device=device, dtype=torch.complex64)).to(dtype)
            v0_vm = torch.abs(torch.tensor(case.v0, device=device, dtype=torch.complex64)).to(dtype)
            config = cupf_gpu.NRConfig()
            config.tolerance = 1e-6
            config.max_iter = 30
            va, vm = cupf_gpu.solve(
                load_p, load_q, solver,
                sbus_base_re=sbus_base_re,
                sbus_base_im=sbus_base_im,
                v0_va=v0_va,
                v0_vm=v0_vm,
                config=config,
                solve_options=cupf_gpu.SolveOptions(),
            )
            loss = va[:, 1].sum() + vm[:, 1].sum()
            loss.backward()
            print('va shape:', tuple(va.shape), 'vm shape:', tuple(vm.shape))
            print('grad finite:', torch.isfinite(load_p.grad).all().item(), torch.isfinite(load_q.grad).all().item())
except Exception as exc:
    print(f'Torch autograd skipped/error: {type(exc).__name__}: {exc}')


Torch autograd skipped: cuPF was not built with Torch extension methods.
